In [ ]:
# piplite 是 Pyodide（浏览器中运行的 Python/WASM 发行版）提供的包管理器，相当于浏览器环境里的 pip
# 它不能在本地 Anaconda/Jupyter 中使用（本地环境没有这个模块，直接运行会报 ModuleNotFoundError）
# 作用：为后面导入 seaborn 做准备
import piplite
# piplite.install() 是异步函数，所以必须用 await 调用（Jupyter 的 cell 顶层支持 await）
# 功能：从 PyPI 下载并安装 seaborn（统计可视化库）到浏览器的虚拟文件系统
# 说明这个 lab 后面会用到 seaborn 的绘图功能（如回归图、分布图）
await piplite.install('seaborn')

# 在本地只需要执行下面步骤
import seaborn  # 本地通常已随 Anaconda 安装

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pyodide.http import pyfetch

# async def：声明异步函数（协程）
# await：等待异步结果，等待时让出执行权
# 适用场景：网络请求、文件下载等等待时间长的 IO 操作——正好符合 notebook 里从网上下载 usedcars.csv 的
# 需
async def download(url, filename):
    response = await pyfetch(url)
    if response.status == 200:
        with open(filename, "wb") as f:
            f.write(await response.bytes())

file_path= "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DA0101EN-SkillsNetwork/labs/Data%20files/automobileEDA.csv"

await download(file_path, "usedcars.csv")
file_name="usedcars.csv"

df = pd.read_csv(file_name)
df.head()

#filepath = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DA0101EN-SkillsNetwork/labs/Data%20files/automobileEDA.csv"
#df = pd.read_csv(filepath, header=None)


# 这行代码从 scikit-learn 库中导入线性回归模型类，是整个 lab 建模的核心工具。
# sklearn	Python 最流行的机器学习库（scikit-learn）
# linear_model	子模块，包含各种线性模型：LinearRegression、Ridge、Lasso 等
# LinearRegression	经典最小二乘线性回归的实现类
# 为什么这样导入，而不是 import sklearn？sklearn 是个庞大的库，直接 import sklearn 不会加载子模块，且用起来要写 sklearn.linear_model.LinearRegression(...)，冗长 按需导入既加快加载速度，代码也更简洁：lm = LinearRegression()
from sklearn.linear_model import LinearRegression

#调用类的构造函数，实例化一个模型
#变量名（linear model 的缩写，只是个名字，可以随便取）
lm = LinearRegression()
# 单独写一个变量名是 Jupyter 的显示技巧：cell 的最后一个表达式会自动打印输出
lm
#公式：Y=a+b*highway-mpg
# 创建模型                      训练            预测             查看参数
# lm = LinearRegression() → lm.fit(X, Y) → lm.predict(X) → lm.intercept_ / lm.coef_
# （空模型）               （学到 a、b）    （算出 Ŷ）        （看到具体数值）

X = df[['highway-mpg']]
Y = df['price']
# 这行是干嘛的？
# fit() 是训练/拟合的核心调用。它做的事：
# 拿到全部样本 
# (xi,yi)：每辆车的 highway-mpg 和价格
# 用最小二乘法求解，找到最优的截距a 和 斜率b，使预测误差平方和最小
lm.fit(X,Y)

#Yhat 输出：预测价格数组（Yhat 即 Ŷ，"Y hat" 是统计学里预测值的标准记号）
#lm 调用模型的预测方法
#输入：自变量（每辆车的 highway-mpg）
# 前提：上一节已经执行过 lm.fit(X, Y)，模型内部已学到了参数，相当于学会了公式及其它参数（用训练好的模型做预测）
#predict() 做的事很简单——把每辆车的 highway-mpg 代入上面这个公式，逐行算出预测价格，返回一个 numpy 数组（长度 = 样本数 201）
Yhat=lm.predict(X)
#Yhat 有 201 个值，全打印太长，所以只看前 5 个
Yhat[0:5]    # numpy 切片：取索引 0,1,2,3,4 的前 5 个预测值（不含 5）
#输出：array([13728.46, 13728.46, 17011.30, 10620.56, 14748.34])

#三种价格对比
# Yhat lm.predict(X)  模型预测价格
# Y    df['price']    真实价格
# Y-Yhat  残差        预测误差

#注意，用fit做出来的模型，是一条直线，是拟合、估计出来的一条规律线，并不是标标准准的
#所以为什么后面要perdict，就是因为要代回去再算一遍，不是为了"重新知道成绩”，而是为了知道：这条规律线离每个人的真实成绩差多远，从而判断这条规律靠不靠谱、能不能用来预测

# intercept_ 没有括号：它是属性不是方法，写成 lm.intercept_() 会报错。
# lm.intercept_     # 38423.31        ← 标量
# lm.coef_          # array([-821.73]) ← 数组（多元回归时有多个系数）
# lm.intercept_ = 读出模型学到的截距 
# a ≈ 38423.31
# 下一行通常是 lm.coef_ 读斜率 
# b ≈ -821.73
lm.intercept_
lm.coef_


